# 🚀 AstroLoc-ML on Google Colab

Open this notebook in Colab, switch to a GPU runtime, then run cells top to bottom.

**Switch to GPU:** `Runtime` → `Change runtime type` → **T4 GPU** (free tier) or **L4 / A100** (Pro).

Pipeline:
1. Clone the repo from GitHub.
2. Install Python deps (Colab already has torch + torchvision).
3. Download the HYG star catalog.
4. Smoke-test the pipeline (~30 s on GPU).
5. Full training run (~10 min on T4, ~3 min on A100).
6. Save the checkpoint to Google Drive so it survives the session.

## 1 · Confirm we have a GPU

In [ ]:
!nvidia-smi || echo 'No GPU. Runtime > Change runtime type > T4 GPU.'

## 2 · Clone the repo

Replace `YOUR_USERNAME/astroloc-ml` with your GitHub repo. For a
private repo use the deploy-token URL form documented in
GitHub `Settings → Developer settings`.

In [ ]:
import os, pathlib
REPO = 'YOUR_USERNAME/astroloc-ml'   # <-- change this
if not pathlib.Path('astroloc-ml').exists():
    !git clone https://github.com/{REPO}.git astroloc-ml
%cd astroloc-ml
!ls

## 3 · Install dependencies

Colab already ships with torch + torchvision + matplotlib + numpy +
pandas. We only need the smaller extras.

In [ ]:
!pip install --quiet astropy seaborn python-dotenv pyyaml piexif

## 4 · Download the HYG catalog

In [ ]:
import pathlib
pathlib.Path('data/catalogs').mkdir(parents=True, exist_ok=True)
if not pathlib.Path('data/catalogs/hygdata_v3.csv').exists():
    !curl -sL -o data/catalogs/hygdata_v3.csv \
      https://raw.githubusercontent.com/astronexus/HYG-Database/main/hyg/CURRENT/hygdata_v41.csv
!ls -lh data/catalogs/

## 5 · Smoke training (sanity check, ~30 s on T4)

In [ ]:
!python train.py --config configs/default.yaml --smoke --skip-phase3 --device cuda

## 6 · Full training run

Edit `configs/default.yaml` first if you want to shrink the dataset —
default is 50,000 synthetic samples × 20 epochs which is ~10 min on
a free T4. Phase 3 is auto-skipped unless `real_data.enabled: true`
and you've put real images in `data/real_images/`.

In [ ]:
!python train.py --config configs/default.yaml --device cuda --skip-phase3 2>&1 | tail -200

## 7 · Inspect the trained checkpoint

In [ ]:
import torch, json
ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
print('Best val angular sep (deg):', ckpt['best_metric'])
print('History epochs:', len(ckpt.get('history', [])))
print(json.dumps(ckpt['history'][-1], indent=2, default=str))

## 8 · Regenerate README figures from the trained checkpoint

In [ ]:
!python scripts/generate_readme_figures.py

In [ ]:
from IPython.display import Image, display
for p in ['reports/figures/05_training_curves.png',
          'reports/figures/09_demo_overlay_0.png',
          'reports/figures/09_demo_overlay_1.png']:
    display(Image(p))

## 9 · Save the checkpoint to Google Drive (so it survives session restarts)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, pathlib, time
out_dir = pathlib.Path('/content/drive/MyDrive/astroloc-ml/checkpoints')
out_dir.mkdir(parents=True, exist_ok=True)
stamp = time.strftime('%Y%m%d_%H%M%S')
shutil.copy('checkpoints/best.pt', out_dir / f'best_{stamp}.pt')
shutil.copy('reports/figures/05_training_curves.png',
            out_dir / f'training_curves_{stamp}.png')
print('Saved to', out_dir)

## 10 · Optional: download checkpoint to your laptop

Colab will pop up a save dialog.

In [ ]:
from google.colab import files
files.download('checkpoints/best.pt')